In [ ]:
from textblob import TextBlob
from sklearn import model_selection,preprocessing,linear_model,naive_bayes,metrics
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn import decomposition,ensemble

import pandas,xgboost,numpy,textblob,string
from keras.preprocessing import text,sequence
from keras import layers,models,optimizers

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np

data=pd.read_excel("dataset.xls")
data=data.dropna()

In [ ]:
df=pd.DataFrame()
df["text"]=data["ACIKLAMA"]
df["label"]=data["ILISKILI_BIRIM"]

In [ ]:
# Camel case Pascal case Transformation:
# B?y?k harf - K???k harf D?n???m?:
df["text"]=df["text"].apply(lambda x: " ".join(x.lower() for x in x.split()))

In [ ]:
# Cleaning Punctuation:
# Noktalama ??aretlerini Silme:
df["text"]=df["text"].str.replace("[^\w\s]","")

In [ ]:
# Cleaning numbers:
# Say?lar? silme:
df["text"]=df["text"].str.replace("\d","")

In [ ]:
# Stemming:
# Kelimeleri k?klerine indirgeme:
from trnlp import TrnlpWord

obj=TrnlpWord()

def stemming(text):
    words=[]
    for i in text.split():
        obj.setword(i)
        words.append(obj.get_stem)
    result=" ".join(words)
    return result
df["text"]=df["text"].apply(stemming)

In [ ]:
# STOPWORDS:
# Etkisiz Kelimeler:
import nltk
from nltk.corpus import stopwords
sw=stopwords.words("turkish")
df["text"]=df["text"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))

In [ ]:
# Deletion of less used word:
# Seyrek kelimelerin silinmesi:
sil=pd.Series(" ".join(df["text"]).split()).value_counts()[-1000:]
df["text"]=df["text"].apply(lambda x: " ".join(x for x in x.split() if x not in sil))

In [ ]:
# LEMMATIZATION:
from textblob import Word
df["text"]=df["text"].apply(lambda x: " ".join([Word(word).lemmatize() for word in x.split()]))

In [ ]:
train_x,test_x,train_y,test_y=model_selection.train_test_split(df["text"],
                                                              df["label"],
                                                              random_state=42)

In [ ]:
train_x.shape

In [ ]:
train_y.shape

In [ ]:
encoder=preprocessing.LabelEncoder()

In [ ]:
train_y=encoder.fit_transform(train_y)
test_y=encoder.fit_transform(test_y)

In [ ]:
test_y[0:20]

# CHARACTERS LEVEL TF - IDF

In [ ]:
tf_idf_char_vectorizer=TfidfVectorizer(analyzer="char",ngram_range=(1,3))
tf_idf_char_vectorizer.fit(train_x)

In [ ]:
x_train_tf_idf_char=tf_idf_char_vectorizer.transform(train_x)
x_test_tf_idf_char=tf_idf_char_vectorizer.transform(test_x)

## ML: LOGISTIC REGRESSION

In [ ]:
loj=linear_model.LogisticRegression()
loj_model=loj.fit(x_train_tf_idf_char,train_x)
accuracy=model_selection.cross_val_score(loj_model,
                                        x_test_tf_idf_char,
                                        test_y,
                                        cv=10).mean()
print("Char TF IDF Do?ruluk Oran?: ",accuracy)

## ML: NAIVE BAYES

In [ ]:
nb=naive_bayes.MultinomialNB()
nb_model=nb.fit(x_train_tf_idf_char,train_y)
accuracy=model_selection.cross_val_score(nb_model,
                                        x_test_tf_idf_char,
                                        test_y,
                                        cv=10).mean()
print("Char TF IDF Do?ruluk Oran?: ",accuracy)

## ML: RANDOM FOREST

In [ ]:
rf=ensemble.RandomForestClassifier()
rf_model=rf.fit(x_train_tf_idf_char,train_y)
accuracy=model_selection.cross_val_score(rf_model,
                                        x_test_tf_idf_char,
                                        test_y,
                                        cv=10).mean()
print("Char TF IDF Do?ruluk Oran?: ",accuracy)

## ML: XGBoost CLASSIFIER

In [ ]:
xgb=xgboost.XGBClassifier()
xgb_model=xgb.fit(x_train_tf_idf_char,train_y)
accuracy=model_selection.cross_val_score(xgb_model,
                                        x_test_tf_idf_char,
                                        test_y,
                                        cv=10).mean()
print("Char TF IDF Do?ruluk Oran?: ",accuracy)

## ML: SVM

In [ ]:
svm=SVC()
svm_model=svm.fit(x_train_tf_idf_char,train_y)
accuracy=model_selection.cross_val_score(svm_model,
                                        x_test_tf_idf_char,
                                        test_y,
                                        cv=10).mean()
print("Char TF IDF Do?ruluk Oran?: ",accuracy)

## ML: KNN

In [ ]:
knn=KNeighborsClassifier()
knn_model=knn.fit(x_train_tf_idf_char,train_y)
accuracy=model_selection.cross_val_score(knn_model,
                                        x_test_tf_idf_char,
                                        test_y,
                                        cv=10).mean()
print("Char TF IDF Do?ruluk Oran?: ",accuracy)

###### Logistic Ragression: 0.7254285714285713

###### Naive Bayes: 0.6973877551020408

# Random Forest: 0.7595102040816326
    
# XGBoost: 0.7575102040816326
    
###### SVM: 0.7375102040816326
    
###### KNN: 0.7455510204081632